In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
import pytest

# Load ingestion configs
ingestion_configs = parse_config_from_json(
    "/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/config/ingestion_configs.json"
)

In [0]:
# Test all expected modules exist in config

def test_config_has_all_modules():
    assert "Products" in ingestion_configs
    assert "Customers" in ingestion_configs
    assert "Orders" in ingestion_configs

def test_config_has_required_keys():
    required = ["file_path", "file_format", "target_table", "schema_alias"]
    for module in ingestion_configs:
        for key in required:
            assert key in ingestion_configs[module], f"{module} missing '{key}'"

In [0]:
# Test file format routing

def test_products_format_is_csv():
    assert ingestion_configs["Products"]["file_format"] == "csv"

def test_orders_format_is_json():
    assert ingestion_configs["Orders"]["file_format"] == "json"

def test_customers_format_is_excel():
    assert ingestion_configs["Customers"]["file_format"] == "excel"

def test_invalid_format_raises():
    with pytest.raises(Exception, match="Invalid file format"):
        fmt = "parquet"
        if fmt not in ("json", "csv", "excel"):
            raise Exception("Invalid file format")

In [0]:
# Test schema alias renames columns correctly - Products

def test_schema_alias_products():
    df = spark.createDataFrame(
        [("P1", "Tech", "Phone", "iPhone", "CA", "999")],
        ["Product ID", "Category", "Sub-Category", "Product Name", "State", "Price per product"]
    )
    aliases = ingestion_configs["Products"]["schema_alias"]
    result = apply_schema_alias(df, aliases)
    expected = ["product_id", "category", "sub_category", "product_name", "state", "price_per_product"]
    assert result.columns == expected

In [0]:
# Test schema alias renames columns correctly - Orders

def test_schema_alias_orders():
    df = spark.createDataFrame(
        [("1", "O1", "1/1/2024", "5/1/2024", "Standard", "C1", "P1", "2", "10", "0.1", "5")],
        ["Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode",
         "Customer ID", "Product ID", "Quantity", "Price", "Discount", "Profit"]
    )
    aliases = ingestion_configs["Orders"]["schema_alias"]
    result = apply_schema_alias(df, aliases)
    expected = ["row_id", "order_id", "order_date", "ship_date", "ship_mode",
                "customer_id", "product_id", "quantity", "price", "discount", "profit"]
    assert result.columns == expected

In [0]:
# Test schema alias renames columns correctly - Customers

def test_schema_alias_customers():
    df = spark.createDataFrame(
        [("C1", "John", "j@x.com", "123", "1 St", "Corp", "US", "NY", "NY", "10001", "East")],
        ["Customer ID", "Customer Name", "email", "phone", "address",
         "Segment", "Country", "City", "State", "Postal Code", "Region"]
    )
    aliases = ingestion_configs["Customers"]["schema_alias"]
    result = apply_schema_alias(df, aliases)
    expected = ["customer_id", "customer_name", "email", "phone", "address",
                "segment", "country", "city", "state", "postal_code", "region"]
    assert result.columns == expected

In [0]:
# Test target tables are correct

def test_target_tables():
    assert ingestion_configs["Products"]["target_table"] == "az_ci_de_dbs.ecom_dev.stg_products"
    assert ingestion_configs["Customers"]["target_table"] == "az_ci_de_dbs.ecom_dev.stg_customers"
    assert ingestion_configs["Orders"]["target_table"] == "az_ci_de_dbs.ecom_dev.stg_orders"

In [0]:
# Test get_value_from_dict returns correct config

def test_get_config_for_existing_module():
    cfg = get_value_from_dict(ingestion_configs, "Products")
    assert cfg["file_format"] == "csv"

def test_get_config_for_missing_module():
    result = get_value_from_dict(ingestion_configs, "NonExistent")
    assert result is None

In [0]:
# Test source file row count matches staging table row count

def test_source_file_row_count_matches_table():
    for module in ingestion_configs:
        cfg = ingestion_configs[module]
        src_path = cfg["file_path"]
        src_format = cfg["file_format"]
        target_table = cfg["target_table"]

        # read source file
        if src_format == "csv":
            source_df = csv_reader(src_path)
        elif src_format == "json":
            source_df = json_reader(src_path)
        elif src_format == "excel":
            sheet = cfg["params"]["sheet_name"]
            source_df = excel_reader(src_path, sheet)

        # read target table
        table_df = spark.read.table(target_table)

        src_count = source_df.count()
        tbl_count = table_df.count()

        assert src_count == tbl_count, (
            f"{module}: source has {src_count} rows but {target_table} has {tbl_count} rows"
        )
        print(f"  {module}: source={src_count}, table={tbl_count} ✓")

In [0]:
# Run all tests
def test_ingestion_main():
    
    test_functions = [
        test_config_has_all_modules,
        test_config_has_required_keys,
        test_products_format_is_csv,
        test_orders_format_is_json,
        test_customers_format_is_excel,
        test_invalid_format_raises,
        test_schema_alias_products,
        test_schema_alias_orders,
        test_schema_alias_customers,
        test_target_tables,
        test_get_config_for_existing_module,
        test_get_config_for_missing_module,
        test_source_file_row_count_matches_table,
    ]

    passed, failed = 0, 0
    for fn in test_functions:
        try:
            fn()
            passed += 1
            print(f"  [PASSED] {fn.__name__}")
        except Exception as e:
            failed += 1
            print(f"  [FAILED] {fn.__name__}: {e}")

    print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")